In [1]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as mpatches
import numpy as np
from scipy import optimize
import importlib
import functions_to_import as fhealthy
import solver.model
importlib.reload(solver.model)
importlib.reload(solver.model.cwrapping)
from sklearn.metrics import mean_squared_error
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
#clang -shared -o path/libModel_Arts_Audebert_Shi.so path/FULL_MODEL_Arts_Audebert_Shi.c -arch x86_64    --->    run in terminal

In [2]:
def default_plot(f,m,index,n_cycles,Tc,tFinal,color,draw):
    
    idx = m.t > tFinal - n_cycles*Tc
    plt.sca(f[0])
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('LV','Pi')[idx],color=color,linestyle='-',linewidth = 1.5)
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('PA_RCL','Pi')[idx],color=color,linestyle='--',linewidth = 1.5)
    if draw:
        time = (m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0])
        diff = m.getVariable('LV','Pi')[idx]-m.getVariable('AA_RCL','Pi')[idx]
        p1 = time[np.where(diff>0)[-1][-1]]
        closed = time[np.where(m.getVariable('LV','Qo')[idx]<0.1)[-1]]
        plt.axvline(time[np.where(diff>0)[-1][-1]],color='b',linestyle='-',linewidth = 1.5)
        plt.axvline(closed[np.where(closed>p1)[-1][0]],color='b',linestyle='-',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Pressure (mmHg)')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    plt.ylim([0,45])
    f[0].set_title('Left ventricle and pulmonary artery')

    plt.sca(f[1])
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('LV','Qo')[idx],color=color,linestyle='-',linewidth = 1.5)
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('LA','Qo')[idx],color=color,linestyle='--',linewidth = 1.5)
    if draw:
        plt.axvline(time[np.where(diff>0)[-1][-1]],color='b',linestyle='--',linewidth = 1.5)
        plt.axvline(closed[np.where(closed>p1)[-1][0]],color='b',linestyle='-',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Blood flow (ml/s)')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    plt.ylim([0,45])
    f[1].set_title('Aortic and mitral valves')

    plt.sca(f[2])
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('RV','Pi')[idx],color=color,linestyle='-',linewidth = 1.5)
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('AA_RCL','Pi')[idx],color=color,linestyle='--',linewidth = 1.5)
    if draw:
        diff2 = m.getVariable('RV','Pi')[idx]-m.getVariable('PA_RCL','Pi')[idx]
        p2 = time[np.where(diff2>0)[-1][-1]]
        closed2 = time[np.where(m.getVariable('RV','Qo')[idx]<0.1)[-1]]
        plt.axvline(time[np.where(diff2>0)[-1][-1]],color='r',linestyle='--',linewidth = 1.5)
        plt.axvline(closed2[np.where(closed2>p2)[-1][0]],color='r',linestyle='-',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Pressure (mmHg)')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    plt.ylim([0,45])
    f[2].set_title('Right ventricle and aorta')

    plt.sca(f[3])
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('RV','Qo')[idx],color=color,linestyle='-',linewidth = 1.5)
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('RA','Qo')[idx],color=color,linestyle='--',linewidth = 1.5)
    if draw:
        plt.axvline(time[np.where(diff2>0)[-1][-1]],color='r',linestyle='--',linewidth = 1.5)
        plt.axvline(closed2[np.where(closed2>p2)[-1][0]],color='r',linestyle='-',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Blood flow (ml/s)')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    plt.ylim([0,45])
    f[3].set_title('Pulmonary and tricuspid valves')

    plt.sca(f[4])
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('AI_RCL', 'Pi')[idx],color=color,linestyle='-',linewidth = 1.5)
    if draw:
        plt.axvline(time[np.where(diff>0)[-1][-1]],color='b',linestyle='--',linewidth = 1.5)
        plt.axvline(time[np.where(diff2>0)[-1][-1]],color='r',linestyle='--',linewidth = 1.5)
        plt.axvline(closed[np.where(closed>p1)[-1][0]],color='b',linestyle='-',linewidth = 1.5)
        plt.axvline(closed2[np.where(closed2>p2)[-1][0]],color='r',linestyle='-',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Pressure (mmHg)')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    plt.ylim([0,45])
    f[4].set_title('Aortic isthmus')

    plt.sca(f[5])
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('AI_RCL', 'Qo')[idx],color=color,linestyle='-',linewidth = 1.5)
    plt.axhline(0,color='k')
    if draw:
        plt.axvline(time[np.where(diff>0)[-1][-1]],color='b',linestyle='--',linewidth = 1.5)
        plt.axvline(time[np.where(diff2>0)[-1][-1]],color='r',linestyle='--',linewidth = 1.5)
        plt.axvline(closed[np.where(closed>p1)[-1][0]],color='b',linestyle='-',linewidth = 1.5)
        plt.axvline(closed2[np.where(closed2>p2)[-1][0]],color='r',linestyle='-',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Blood flow (ml/s)')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    f[5].set_title('Aortic isthmus')

    plt.sca(f[6])
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('Bif_PAin_LUNG1out_DA2out', 'Pi')[idx],color=color,linestyle='-',linewidth = 1.5)
    if draw:
        plt.axvline(time[np.where(diff>0)[-1][-1]],color='b',linestyle='--',linewidth = 1.5)
        plt.axvline(time[np.where(diff2>0)[-1][-1]],color='r',linestyle='--',linewidth = 1.5)
        plt.axvline(closed[np.where(closed>p1)[-1][0]],color='b',linestyle='-',linewidth = 1.5)
        plt.axvline(closed2[np.where(closed2>p2)[-1][0]],color='r',linestyle='-',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Pressure (mmHg)')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    plt.ylim([0,45])
    f[6].set_title('Ductus arteriosus')

    plt.sca(f[7])
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('Bif_PAin_LUNG1out_DA2out', 'Qo2')[idx],color=color,linestyle='-',linewidth = 1.5)
    if draw:
        plt.axvline(time[np.where(diff>0)[-1][-1]],color='b',linestyle='--',linewidth = 1.5)
        plt.axvline(time[np.where(diff2>0)[-1][-1]],color='r',linestyle='--',linewidth = 1.5)
        plt.axvline(closed[np.where(closed>p1)[-1][0]],color='b',linestyle='-',linewidth = 1.5)
        plt.axvline(closed2[np.where(closed2>p2)[-1][0]],color='r',linestyle='-',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Blood flow (ml/s)')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    f[7].set_title('Ductus arteriosus')

    plt.sca(f[8])
    P_LICA = m.getVariable('ICaroAL_RCL', 'Pi')[idx]
    P_RICA = m.getVariable('ICaroAR_RCL', 'Pi')[idx]
    P_MCA = (P_LICA + P_RICA)/2
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),P_MCA,color=color,linestyle='-',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Pressure (mmHg)')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    plt.ylim([0,45])
    f[8].set_title('Middle cerebral artery')

    plt.sca(f[9])
    Q_LICA = m.getVariable('ICaroAL_RCL', 'Qo')[idx]
    Q_RICA = m.getVariable('ICaroAR_RCL', 'Qo')[idx]
    Q_MCA = (Q_LICA + Q_RICA)*0.75*0.74312
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),Q_MCA,color=color,linestyle='-',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Blood flow (ml/s)')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    f[9].set_title('Middle cerebral artery')

    plt.sca(f[10])
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('UA_RC', 'Pi')[idx],color=color,linestyle='-',linewidth = 1.5)
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('Bif_UVin_DV1out_HE2out', 'Pi')[idx],color=color,linestyle='--',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Pressure (mmHg)')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    f[10].set_title('Umbilical artery and vein')

    plt.sca(f[11])
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('UA_RC', 'Qo')[idx],color=color,linestyle='-',linewidth = 1.5)
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('UV_RCL', 'Qo')[idx],color=color,linestyle='--',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Blood flow (ml/s)')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    f[11].set_title('Umbilical artery and vein')
    
    plt.subplots_adjust(hspace=0.5,wspace=0.4)
    
    

In [3]:
def dv_plot(f,m,index,n_cycles,Tc,tFinal,color,draw):
    
    idx = m.t > tFinal - n_cycles*Tc

    plt.sca(f[0])
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('Bif_DV1in_IVC2in_FOout', 'Qo')[idx],color=color,linestyle='-',linewidth = 1.5)
    if draw:
        plt.axvline(closed[np.where(closed>p1)[-1][0]],color='b',linestyle='--',linewidth = 1.5)
        plt.axvline(time[np.where(diff>0.01)[0][0]]+0.5,color='r',linestyle='--',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Blood flow (ml/s)')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    #plt.ylim([0,5])
    f[0].set_title('FO')


    plt.sca(f[1])
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('LV', 'Ls')[idx],color=color,linestyle='-',linewidth = 1.5)
    if draw:
        plt.axvline(closed[np.where(closed>p1)[-1][0]],color='b',linestyle='--',linewidth = 1.5)
        plt.axvline(time[np.where(diff>0.01)[0][0]]+0.5,color='r',linestyle='--',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Ls')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    plt.ylim([1.75,2.10])
    f[1].set_title('LV')

    plt.sca(f[2])
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('LA', 'Pi')[idx],color=color,linestyle='-',linewidth = 1.5)
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('RA', 'Pi')[idx],color=color,linestyle='--',linewidth = 1.5)
    #plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('LA', 'Pi')[idx],color=color,linestyle='--',linewidth = 1.5)
    if draw:
        plt.axvline(closed[np.where(closed>p1)[-1][0]],color='b',linestyle='--',linewidth = 1.5)
        plt.axvline(time[np.where(diff>0.01)[0][0]]+0.5,color='r',linestyle='--',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Pressure (mmHg)')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    plt.ylim([0,5])
    f[2].set_title('LA and RA')

    plt.sca(f[3])
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('LA', 'Vcav')[idx],color=color,linestyle='-',linewidth = 1.5)
    plt.plot((m.t[idx]-m.t[idx][0])/(m.t[idx][-1]-m.t[idx][0]),m.getVariable('RA', 'Vcav')[idx],color=color,linestyle='--',linewidth = 1.5)
    if draw:
        plt.axvline(closed[np.where(closed>p1)[-1][0]],color='b',linestyle='--',linewidth = 1.5)
        plt.axvline(time[np.where(diff>0.01)[0][0]]+0.5,color='r',linestyle='--',linewidth = 1.5)
    plt.xlabel('Normalised cycle time')
    plt.ylabel('Volume (ml)')
    plt.grid(color = [0.7,0.7,0.7,1])
    plt.xlim([0,1])
    plt.ylim([0,5])
    f[3].set_title('LA and RA')

    
    plt.subplots_adjust(hspace=0.5,wspace=0.4)
    

In [4]:
def ventricular_disproportion_vasculature(m,degree_aovalve,degree_pvvalve,default_values):
    # Radius arch
    radius_bct = 0.19899999999999998
    radius_lsca = 0.136435
    radius_rsca = 0.136435
    radius_lcca = 0.15234000000000003
    radius_ist = 0.2329
    radius_dAoA = 0.24756422700788266
    radius_aAoA = 0.26546686273838843
    radius_aAo = 0.2984690662736599
    radius_tAo = 0.25955
    radius_duct = 0.2090187021180712
    radius_pA = 0.3415

    #print(f"degree_pvvalve = {degree_pvvalve}")
    #print(f"degree_aovalve = {degree_aovalve}")
    
    # Pulmonary artery
    degree_pa = np.sqrt(degree_pvvalve)
    print(f"degree_pa = {degree_pa}")
    m.setConstantOrState0('Parameters', 'Rpa', default_values['Rpa']/(degree_pa**4)) 
    m.setConstantOrState0('Parameters', 'Bmpa', default_values['Bmpa']/(degree_pa**4)) 
    m.setConstantOrState0('Parameters', 'Cpa', default_values['Cpa']*(degree_pa**2))
    m.setConstantOrState0('Parameters', 'Lpa', default_values['Lpa']/(degree_pa**2)) 
    
    # Ascending aorta
    #print(degree_aovalve)
    degree_aAo = np.sqrt(degree_aovalve)
    print(f"degree_aAo = {degree_aAo}")
    m.setConstantOrState0('Parameters', 'Raa', default_values['Raa']/(degree_aAo**4)) 
    m.setConstantOrState0('Parameters', 'Caa', default_values['Caa']*(degree_aAo**2))
    m.setConstantOrState0('Parameters', 'Laa', default_values['Laa']/(degree_aAo**2)) 

    # Ascending aortic arch
    degree_aAoA = np.cbrt(((degree_aAo*radius_aAo)**3 - radius_bct**3)/radius_aAoA**3)
    m.setConstantOrState0('Parameters', 'RaAoA', default_values['RaAoA']/(degree_aAoA**4))     
    m.setConstantOrState0('Parameters', 'CaAoA', default_values['CaAoA']*(degree_aAoA**2))
    m.setConstantOrState0('Parameters', 'LaAoA', default_values['LaAoA']/(degree_aAoA**2))   
        
    # Desdending aortic arch
    degree_dAoA = np.cbrt(((degree_aAoA*radius_aAoA)**3 - radius_lcca**3)/radius_dAoA**3)
    m.setConstantOrState0('Parameters', 'RdAoA', default_values['RdAoA']/(degree_dAoA**4))
    m.setConstantOrState0('Parameters', 'BdAoA', default_values['BdAoA']/(degree_dAoA**4))
    m.setConstantOrState0('Parameters', 'CdAoA', default_values['CdAoA']*(degree_dAoA**2))
    m.setConstantOrState0('Parameters', 'LdAoA', default_values['LdAoA']/(degree_dAoA**2)) 
    
    # Aortic isthmus
    degree_ist = np.cbrt(((degree_dAoA*radius_dAoA)**3 - radius_lsca**3)/radius_ist**3)
    m.setConstantOrState0('Parameters', 'Risthm', default_values['Risthm']/(degree_ist**4))
    m.setConstantOrState0('Parameters', 'Bisthm', default_values['Bisthm']/(degree_ist**4))
    m.setConstantOrState0('Parameters', 'Cisthm', default_values['Cisthm']*(degree_ist**2))
    m.setConstantOrState0('Parameters', 'Listhm', default_values['Listhm']/(degree_ist**2))
    #print(degree_ist)
    


In [5]:
def ductus_change(m,degree_da,default_values):
    m.setConstantOrState0('Parameters', 'Rda', default_values['Rda']/(degree_da**4)) 
    m.setConstantOrState0('Parameters', 'Bda', default_values['Bda']/(degree_da**4))

def ua_change(m,degree_ua,default_values):
    m.setConstantOrState0('Parameters', 'Rua', default_values['Rua']/(degree_ua**4)) 
    m.setConstantOrState0('Parameters', 'Cua', default_values['Cua']*(degree_ua**2)) 
    
def brain_change(m,degree_brain,default_values):
    degree_vessels = np.cbrt(1/degree_brain)
    m.setConstantOrState0('Parameters', 'R_L_ICA', default_values['R_L_ICA']/(degree_vessels**4)) 
    m.setConstantOrState0('Parameters', 'R_L_CCA', default_values['R_L_CCA']/(degree_vessels**4)) 
    m.setConstantOrState0('Parameters', 'R_R_ICA', default_values['R_R_ICA']/(degree_vessels**4)) 
    m.setConstantOrState0('Parameters', 'R_R_CCA', default_values['R_R_CCA']/(degree_vessels**4)) 
    m.setConstantOrState0('Parameters', 'C_L_ICA', default_values['C_L_ICA']*(degree_vessels**2)) 
    m.setConstantOrState0('Parameters', 'C_L_CCA', default_values['C_L_CCA']*(degree_vessels**2)) 
    m.setConstantOrState0('Parameters', 'C_R_ICA', default_values['C_R_ICA']*(degree_vessels**2)) 
    m.setConstantOrState0('Parameters', 'C_R_CCA', default_values['C_R_CCA']*(degree_vessels**2)) 
    m.setConstantOrState0('Parameters', 'Rsvc', default_values['Rsvc']/(degree_vessels**4)) 
    m.setConstantOrState0('Parameters', 'Csvc', default_values['Csvc']*(degree_vessels**2))


In [7]:
import os
import glob
import re

folder1 = "./Results_TGA_100/"
folder2 = "./Flow_TGA_100/"
file_pattern1 = f"FO*_DA*_LUNG*_DV*_BRAIN*.npz" 
file_list = glob.glob(folder1 + file_pattern1)

v_dv = [1,0.9,0.8,0.7,0.6,0.5] 

regex = re.compile(r"FO(\d+)_DA(\d+)_LUNG(\d+)_DV(\d+)_BRAIN(\d+).npz$", re.IGNORECASE)

for ii in range(len(file_list)):
    file_path1 = file_list[ii]
    data = np.load(file_path1)
    t = data['voi']  
    results = data['data']  
    names = data['names']  
    CO = np.trapz(results[:,np.where(np.all(names == ('LV', 'Qo'), axis=1))[0][0]],t)/2 + np.trapz(results[:,np.where(np.all(names == ('RV', 'Qo'), axis=1))[0][0]],t)/2

    file_path2 = folder2 + "Flow_" + os.path.basename(file_path1)
    
    match = regex.search(file_path2)
    FO, DA, LUNG, DV, BRAIN = match.groups()
    input_variables_o2 = np.load(file_path2)
    input_variables_o2_dict = dict(input_variables_o2)
    if input_variables_o2_dict.get("Q_IVCfo", 0) < 0:
        print('damn it')
        input_variables_o2_dict["Q_DVfo"] = input_variables_o2_dict["Q_DVfo"] + input_variables_o2_dict["Q_IVCfo"] - 0.001
        input_variables_o2_dict["Q_IVCfo"] = 0.001
    
    for jj in range(len(v_dv)):
        Q_DV = input_variables_o2_dict["Q_DVfo"] + input_variables_o2_dict["Q_DVra"]
        Q_FO = input_variables_o2_dict["Q_DVfo"] + input_variables_o2_dict["Q_IVCfo"]
        Q_DVfo = Q_DV*v_dv[jj]
        Q_DVra = Q_DV-Q_DVfo
        Q_SVC = input_variables_o2_dict["Q_BR"] + input_variables_o2_dict["Q_UB"]
            
        if jj == 0:
            Q_FO_90 = Q_DVfo + input_variables_o2_dict["Q_IVCfo"]
            
        Q_SVCfo = max(0.00001, Q_FO_90 - Q_DVfo - input_variables_o2_dict["Q_IVCfo"])
        Q_SVCra = Q_SVC - Q_SVCfo
        
        
        m = solver.model.ModelSolver('TGA_O2_model.c')
        total_consumption = 0.22
        
        decrease_brain = (-2.67*float(BRAIN) + 367)/100 #(-4.67*float(BRAIN) + 567)/100
        print(f"{float(BRAIN)} = {decrease_brain}")

        with fhealthy.HiddenPrints():
            tFinal = 100
            [m.setConstantOrState0('Parameters', key, value) for key, value in input_variables_o2.items()]
            m.setConstantOrState0('Parameters', 'Q_SVCfo', Q_SVCfo)       
            m.setConstantOrState0('Parameters', 'Q_SVCra', Q_SVCra)            
            m.setConstantOrState0('Parameters', 'Q_DVfo', Q_DVfo)       
            m.setConstantOrState0('Parameters', 'Q_DVra', Q_DVra)       
            m.setConstantOrState0('Parameters', 'QM_CORONARIES', 0.08*total_consumption)            # 8%
            m.setConstantOrState0('Parameters', 'QM_BR', 0.57*decrease_brain*total_consumption)     # 57%
            m.setConstantOrState0('Parameters', 'QM_UB', 0.024*total_consumption)                   # 2.4% 
            m.setConstantOrState0('Parameters', 'QM_KID', 0.08*total_consumption)                   # 8%
            m.setConstantOrState0('Parameters', 'QM_LEG', 0.07*total_consumption)                   # 7%
            m.setConstantOrState0('Parameters', 'QM_LUNG', 0.08*100/float(LUNG)*total_consumption)  # 8%
            m.setConstantOrState0('Parameters', 'QM_INTE', 0.016*total_consumption)                 # 1.6%
            m.setConstantOrState0('Parameters', 'QM_HE', 0.08*total_consumption)                    # 8%
            
            m.solve(tFinal = tFinal) 

            m.saveResults2(f'./O2_TGA_100/O2_from_FO{FO}_DA{DA}_LUNG{LUNG}_DV{int(v_dv[jj]*100)}_BRAIN{BRAIN}.npz')
        
        print(f"shunting = {v_dv[jj]}")
        print(f"SO2 AA: {np.round(m.getVariable('RV', 'SO2')[-1],3)}%")
        print(f"SO2 PA: {np.round(m.getVariable('LV', 'SO2')[-1],3)}%")
        print(f"SO2 dAo: {np.round(m.getVariable('Bif_AO1in_DA2in_dAOout', 'SO2')[-1],3)}%")
        print(f"SO2 UV: {np.round(m.getVariable('PLAC', 'SO2')[-1],3)}%")
        print(f"SO2 LUNG: {np.round(m.getVariable('LUNG', 'SO2')[-1],3)}%")
        print(f"SO2 IVC: {np.round(m.getVariable('Bif_KID1in_HE2in_LEG3in_IVCout', 'SO2')[-1],3)}%")
        print(f"SO2 SVC: {np.round(m.getVariable('Bif_BR1in_UB2in_SVCout', 'SO2')[-1],3)}%")
        print(f"SO2 DV: {np.round(m.getVariable('Bif_UVin_DV1out_IntraHe2out', 'SO2_1')[-1],3)}%")
        
        
        print(f"COO: {np.trapz(results[:,np.where(np.all(names == ('RV', 'Qo'), axis=1))[0][0]],t)/2 + np.trapz(results[:,np.where(np.all(names == ('LV', 'Qo'), axis=1))[0][0]],t)/2}")
        print(f"CO dAo: {np.trapz(results[:,np.where(np.all(names == ('AO2_RCL', 'Qo'), axis=1))[0][0]],t)/2}")
        print(f"CO UV: {np.trapz(results[:,np.where(np.all(names == ('UV_RCL', 'Qo'), axis=1))[0][0]],t)/2}")
        print(f"Q_DV = {Q_DV}")
        print(f"Q_DVfo = {Q_DVfo}")
        print(f"Q_IVCfo = {input_variables_o2_dict['Q_IVCfo']}")
        print(f"Q_SVCfo = {Q_SVCfo}") #Bif_DVin_FO1out_RA2out
        print(f"Q_LUNG = {input_variables_o2_dict['Q_LUNG']}")
        
        
        print(f"cO2o IVC: {np.round(m.getVariable('Bif_KID1in_HE2in_LEG3in_IVCout', 'cO2o')[-1],3)}")
        print(f"cO2o SVC: {np.round(m.getVariable('Bif_BR1in_UB2in_SVCout', 'cO2o')[-1],3)}")
        print(f"cO2o DV: {np.round(m.getVariable('Bif_UVin_DV1out_IntraHe2out', 'cO2o1')[-1],3)}")
        print(f"cO2o LUNG: {np.round(m.getVariable('LUNG', 'cO2o')[-1],3)}")


running build_ext
building 'computeRates' extension
clang -Wno-unused-result -Wsign-compare -Wunreachable-code -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /opt/anaconda3/include -arch x86_64 -I/opt/anaconda3/include -fPIC -O2 -isystem /opt/anaconda3/include -arch x86_64 -I/opt/anaconda3/lib/python3.9/site-packages/numpy/core/include -I/opt/anaconda3/include/python3.9 -c computeRates.c -o build/temp.macosx-10.9-x86_64-cpython-39/computeRates.o --std=c99


In file included from computeRates.c:773:
In file included from /opt/anaconda3/lib/python3.9/site-packages/numpy/core/include/numpy/arrayobject.h:5:
In file included from /opt/anaconda3/lib/python3.9/site-packages/numpy/core/include/numpy/ndarrayobject.h:12:
In file included from /opt/anaconda3/lib/python3.9/site-packages/numpy/core/include/numpy/ndarraytypes.h:1948:
/opt/anaconda3/lib/python3.9/site-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: "Using deprecated NumPy API, disable it with "          "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-W#warnings]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^
1 warning generated.


clang -Wno-unused-result -Wsign-compare -Wunreachable-code -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /opt/anaconda3/include -arch x86_64 -I/opt/anaconda3/include -fPIC -O2 -isystem /opt/anaconda3/include -arch x86_64 -I/opt/anaconda3/lib/python3.9/site-packages/numpy/core/include -I/opt/anaconda3/include/python3.9 -c model.c -o build/temp.macosx-10.9-x86_64-cpython-39/model.o --std=c99
clang -bundle -undefined dynamic_lookup -Wl,-rpath,/opt/anaconda3/lib -L/opt/anaconda3/lib -L/opt/anaconda3/lib -Wl,-rpath,/opt/anaconda3/lib -L/opt/anaconda3/lib build/temp.macosx-10.9-x86_64-cpython-39/computeRates.o build/temp.macosx-10.9-x86_64-cpython-39/model.o -o build/lib.macosx-10.9-x86_64-cpython-39/computeRates.cpython-39-darwin.so


ld: warning: duplicate -rpath '/opt/anaconda3/lib' ignored


copying build/lib.macosx-10.9-x86_64-cpython-39/computeRates.cpython-39-darwin.so -> 
110.0 = 0.7330000000000001
shunting = 1
SO2 AA: 49.41%
SO2 PA: 65.231%
SO2 dAo: 57.325%
SO2 UV: 79.28%
SO2 LUNG: 61.953%
SO2 IVC: 62.802%
SO2 SVC: 41.184%
SO2 DV: 79.28%
COO: 6.6385817527771
CO dAo: 2.5639636516571045
CO UV: 1.5933964252471924
Q_DV = 1.017
Q_DVfo = 1.017
Q_IVCfo = 2.73
Q_SVCfo = 1e-05
Q_LUNG = 2.344
cO2o IVC: 0.144
cO2o SVC: 0.094
cO2o DV: 0.182
cO2o LUNG: 0.142
running build_ext
copying build/lib.macosx-10.9-x86_64-cpython-39/computeRates.cpython-39-darwin.so -> 
110.0 = 0.7330000000000001
shunting = 0.9
SO2 AA: 50.419%
SO2 PA: 64.224%
SO2 dAo: 57.325%
SO2 UV: 79.28%
SO2 LUNG: 60.946%
SO2 IVC: 62.802%
SO2 SVC: 42.192%
SO2 DV: 79.28%
COO: 6.6385817527771
CO dAo: 2.5639636516571045
CO UV: 1.5933964252471924
Q_DV = 1.017
Q_DVfo = 0.9152999999999999
Q_IVCfo = 2.73
Q_SVCfo = 0.10170000000000012
Q_LUNG = 2.344
cO2o IVC: 0.144
cO2o SVC: 0.097
cO2o DV: 0.182
cO2o LUNG: 0.14
running build_ext

...

ld: warning: duplicate -rpath '/opt/anaconda3/lib' ignored


copying build/lib.macosx-10.9-x86_64-cpython-39/computeRates.cpython-39-darwin.so -> 
100.0 = 1.0
shunting = 0.7
SO2 AA: 50.719%
SO2 PA: 61.411%
SO2 dAo: 55.361%
SO2 UV: 80.866%
SO2 LUNG: 58.218%
SO2 IVC: 62.519%
SO2 SVC: 36.04%
SO2 DV: 80.866%
COO: 6.333247423171997
CO dAo: 2.628920078277588
CO UV: 1.6333575248718262
Q_DV = 1.002
Q_DVfo = 0.7013999999999999
Q_IVCfo = 2.333
Q_SVCfo = 0.30059999999999976
Q_LUNG = 2.705
cO2o IVC: 0.143
cO2o SVC: 0.083
cO2o DV: 0.185
cO2o LUNG: 0.133
running build_ext
copying build/lib.macosx-10.9-x86_64-cpython-39/computeRates.cpython-39-darwin.so -> 
100.0 = 1.0
shunting = 0.6
SO2 AA: 51.666%
SO2 PA: 60.178%
SO2 dAo: 55.361%
SO2 UV: 80.866%
SO2 LUNG: 56.984%
SO2 IVC: 62.519%
SO2 SVC: 36.988%
SO2 DV: 80.866%
COO: 6.333247423171997
CO dAo: 2.628920078277588
CO UV: 1.6333575248718262
Q_DV = 1.002
Q_DVfo = 0.6012
Q_IVCfo = 2.333
Q_SVCfo = 0.4007999999999998
Q_LUNG = 2.705
cO2o IVC: 0.143
cO2o SVC: 0.085
cO2o DV: 0.185
cO2o LUNG: 0.13
running build_ext
build

In file included from computeRates.c:773:
In file included from /opt/anaconda3/lib/python3.9/site-packages/numpy/core/include/numpy/arrayobject.h:5:
In file included from /opt/anaconda3/lib/python3.9/site-packages/numpy/core/include/numpy/ndarrayobject.h:12:
In file included from /opt/anaconda3/lib/python3.9/site-packages/numpy/core/include/numpy/ndarraytypes.h:1948:
/opt/anaconda3/lib/python3.9/site-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: "Using deprecated NumPy API, disable it with "          "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-W#warnings]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^
1 warning generated.


clang -Wno-unused-result -Wsign-compare -Wunreachable-code -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /opt/anaconda3/include -arch x86_64 -I/opt/anaconda3/include -fPIC -O2 -isystem /opt/anaconda3/include -arch x86_64 -I/opt/anaconda3/lib/python3.9/site-packages/numpy/core/include -I/opt/anaconda3/include/python3.9 -c model.c -o build/temp.macosx-10.9-x86_64-cpython-39/model.o --std=c99
clang -bundle -undefined dynamic_lookup -Wl,-rpath,/opt/anaconda3/lib -L/opt/anaconda3/lib -L/opt/anaconda3/lib -Wl,-rpath,/opt/anaconda3/lib -L/opt/anaconda3/lib build/temp.macosx-10.9-x86_64-cpython-39/computeRates.o build/temp.macosx-10.9-x86_64-cpython-39/model.o -o build/lib.macosx-10.9-x86_64-cpython-39/computeRates.cpython-39-darwin.so
copying build/lib.macosx-10.9-x86_64-cpython-39/computeRates.cpython-39-darwin.so -> 
100.0 = 1.0
shunting = 0.5
SO2 AA: 52.574%
SO2 PA: 58.995%
SO2 dAo: 55.361%
SO2 UV: 80.866%
SO2 LUNG: 55.801%
SO2 IVC: 62.519%
SO2 SVC: 37.896%
SO2 DV: 80.866%
COO: 6.333247423

ld: warning: duplicate -rpath '/opt/anaconda3/lib' ignored
